In [4]:
from google.colab import files
uploaded = files.upload()


Saving russia_wc_2016.json to russia_wc_2016.json


In [5]:
import json
import csv
import os

class NoSuchFieldError(Exception):
    def __init__(self, message):
        super().__init__(message)

class IllegalArgumentError(ValueError):
    pass

def load_players(filename):
    """Загрузить данные о хоккеистах из json-файла 'filename'."""
    with open(filename, 'r', encoding='utf-8') as file:
        raw_data = json.load(file)

    players = []
    for p in raw_data:
        clean_p = {k.strip(): (v.strip() if isinstance(v, str) else v) for k, v in p.items()}
        players.append(clean_p)

    for player in players:
        player["год_рождения"] = int(player["дата_рождения"].split("-")[0])

        height_m = player["рост"] / 100.0
        imt = player["вес"] / (height_m ** 2)
        player["ИМТ"] = imt

        if imt <= 16:
            player["ИМТ_категория"] = "Выраженный дефицит массы тела"
        elif imt < 18.5:
            player["ИМТ_категория"] = "Недостаточная масса тела"
        elif imt < 25:
            player["ИМТ_категория"] = "Норма"
        elif imt < 30:
            player["ИМТ_категория"] = "Избыточная масса тела"
        elif imt < 35:
            player["ИМТ_категория"] = "Ожирение первой степени"
        elif imt < 40:
            player["ИМТ_категория"] = "Ожирение второй степени"
        else:
            player["ИМТ_категория"] = "Ожирение третьей степени"

    return players

def group_by(players, field):
    """Вернуть данные, сгруппированные по полю 'field'."""
    res = {}

    for player in players:
        if field not in player:
            raise NoSuchFieldError(f"Нет данных для игрока по полю '{field}'")
        value = player[field]
        res[value] = res.get(value, 0) + 1

    return res

def save_group_data(filename, group_data, headers):
    """Сохранить данные 'group_data' в csv-файл 'filename' с заголовками 'headers'."""
    if not isinstance(headers, list) or len(headers) != 2:
        raise IllegalArgumentError("'headers' не список или не содержит ровно 2 элемента.")

    sorted_items = sorted(group_data.items(), key=lambda x: (-x[1], x[0]))

    with open(filename, 'w', encoding='utf-8', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(headers)
        for key, count in sorted_items:
            writer.writerow([key, count])

if __name__ == "__main__":
    filename = input("Введите имя JSON-файла (Enter для 'russia_wc_2016.json'): ").strip()
    if not filename:
        filename = "russia_wc_2016.json"

    save_filename = input("Введите имя CSV-файла (Enter для 'output.csv'): ").strip()
    if not save_filename:
        save_filename = "output.csv"

    field = input("Введите поле для группировки (имя/амплуа/рука/год_рождения/клуб/клуб_страна/ИМТ_категория): ").strip()
    if not field:
        field = "амплуа"

    try:
        players = load_players(filename)
        print(f"Загружено {len(players)} игроков")

        group_data = group_by(players, field=field)
        print(f"Сгруппировано по полю '{field}': {len(group_data)} групп")

        save_group_data(save_filename, group_data, headers=[field.capitalize(), "Количество"])
        print(f"Данные сохранены в файл '{save_filename}'")

    except FileNotFoundError:
        print(f"Ошибка: файл '{filename}' не найден")
    except NoSuchFieldError as e:
        print(f"Ошибка структуры данных: {e}")
    except IllegalArgumentError as e:
        print(f"Ошибка аргументов: {e}")
    except Exception as e:
        print(f"Произошла ошибка: {e}")

Введите имя JSON-файла (Enter для 'russia_wc_2016.json'): 
Введите имя CSV-файла (Enter для 'output.csv'): 
Введите поле для группировки (имя/амплуа/рука/год_рождения/клуб/клуб_страна/ИМТ_категория): имя
Загружено 25 игроков
Сгруппировано по полю 'имя': 18 групп
Данные сохранены в файл 'output.csv'
